# Acceptance: `cumulative_integral` and `cumsum`

**This is a sign-off page, not a tutorial.** It checks the split of
the old `Output.cumulative()` into two methods (plan `plans/cumulative-integral.plan.md`).

`cumulative()` was a plain running sum along time. That is a cumulative count
only when each saved value is already an amount for its interval. A saved flow
(`FlowMass`) is a **rate**, people per day, and a running sum of a rate is
people only when saves are exactly one day apart, and even then it counts the
first sample as a whole day. Two explicit methods replace it:

| Method | Use it on | Returns |
|---|---|---|
| `cumulative_integral(start=, end=)` | a rate, such as a `FlowMass` | trapezoid integral from `start` to each save time; same length, 0 at `start` |
| `cumsum(start=, end=)` | amounts per interval: `integrate_intervals()` output, `midpoint()` series, counts | a running sum of the saved values |

The model is a closed SIR. Every infection leaves S, so the fall in S is the
true cumulative count of infections, solved with a tight-tolerance `tsit5`.
Saves are **two days apart**, so a running sum of the rate is visibly wrong.

Nothing here asserts; the checks are in `tests/test_cumulative_integral.py`. Run
every cell top to bottom (under a minute). Each section ends with a **What to
check** box.

| | You are checking that… |
|---|---|
| A | `cumulative_integral()` of the infection rate follows the fall in S |
| B | that holds for any save spacing |
| C | `cumulative_integral()` is `integrate_intervals().cumsum()` with a leading 0, and `start=` restarts it from 0 |
| D | the traced program does not grow with the number of save times |


In [ ]:
import jax
import jax.numpy as jnp
import numpy as np
import pandas as pd
import plotly.io as pio

pd.options.plotting.backend = "plotly"
pio.renderers.default = "notebook_connected"

from summer4 import FlowMass, FlowModel, OutputSet, Param, Property, PropertyMap, SavePlan, TransitionFlow
from summer4.epi import ForceOfInfection
from summer4.results.output import Output
from summer4.time import TimeAxis

state = Property("state", ("S", "I", "R"))
model = FlowModel(PropertyMap.from_property(state))
model.add_flow(
    TransitionFlow(
        "infection",
        state["S"],
        state["I"],
        ForceOfInfection("foi", infectious=state["I"], contact_rate=Param("beta")),
    )
)
model.add_flow(TransitionFlow("recovery", state["I"], state["R"], 0.1))
cm = model.compile()
y0 = np.array([990.0, 10.0, 0.0])
TSIT5 = dict(solver="tsit5", rtol=1e-8, atol=1e-8)


def solve(spacing):
    """Solve to day 60, saving every `spacing` days."""
    return cm.run({"beta": 0.3}, y0, t0=0.0, t1=60.0, dt=spacing, **TSIT5)


res = solve(2.0)
rate = res["infection"].sum_by()
susceptible = np.asarray(res["compartments"].select(state["S"]).sum_by().values)
days = pd.Index(np.asarray(rate.times.values), name="day")
rate.dims, days[:4].tolist()

## A. Cumulative infections, three ways

`rate` is new infections per day at each save time (every 2 days). The figure
draws the true count, `S(0) - S(t)`, and the `cumulative_integral()` of the rate.

In [ ]:
frame_a = pd.DataFrame(
    {
        "S(0) - S(t): true cumulative infections": susceptible[0] - susceptible,
        "rate.cumulative_integral()": np.asarray(rate.cumulative_integral().values),
    },
    index=days,
)
frame_a.plot(
    title="A. Cumulative infections with saves every 2 days",
    labels={"value": "people", "variable": ""},
).show()
frame_a.iloc[[0, 1, 5, 10, -1]].round(1)

> **What to check (A).** The `cumulative_integral()` curve lies on the true
> count, and the table's last row agrees to within about 0.1%. Both start at 0
> on day 0.

## B. Any save spacing

The same solve, saved every 0.5, 1, 2 and 5 days. For each spacing the bars are
the final count from the fall in S and from `cumulative_integral()`.

In [ ]:
rows = []
for spacing in (0.5, 1.0, 2.0, 5.0):
    r = solve(spacing)
    s = np.asarray(r["compartments"].select(state["S"]).sum_by().values)
    f = r["infection"].sum_by()
    rows.append(
        {
            "save spacing (days)": spacing,
            "S(0) - S(end)": float(s[0] - s[-1]),
            "cumulative_integral()": float(f.cumulative_integral().values[-1]),
        }
    )
frame_b = pd.DataFrame(rows).set_index("save spacing (days)")
frame_b.plot.bar(
    barmode="group",
    title="B. Final cumulative infections by save spacing",
    labels={"value": "people", "variable": ""},
).show()
frame_b.round(1)

> **What to check (B).** At every spacing the `cumulative_integral()` bar matches
> the `S(0) - S(end)` bar (within about 1% at 5 days, where the trapezoid rule
> is coarsest).

## C. The same numbers as the two-step idiom, and a window

`integrate_intervals()` turns the rate into people per save interval (one fewer
row, timed at each interval's right edge); `cumsum()` then adds those amounts
up. `cumulative_integral()` should equal that with a 0 in front. The second
figure starts the count on day 20, a save time: 0 before and on day 20, then a
running integral.

In [ ]:
two_step = rate.integrate_intervals().cumsum()
frame_c = pd.DataFrame(
    {
        "rate.cumulative_integral()": np.asarray(rate.cumulative_integral().values),
        "rate.integrate_intervals().cumsum(), 0 in front": np.concatenate(
            [[0.0], np.asarray(two_step.values)]
        ),
    },
    index=days,
)
frame_c.plot(
    title="C. cumulative_integral() against integrate_intervals().cumsum()",
    labels={"value": "people", "variable": ""},
).show()

from_day_20 = rate.cumulative_integral(start=20.0)
pd.DataFrame(
    {
        "cumulative_integral(start=20)": np.asarray(from_day_20.values),
        "S(20) - S(t), from day 20": np.where(
            days >= 20.0, susceptible[days.get_loc(20.0)] - susceptible, 0.0
        ),
    },
    index=days,
).plot(
    title="C. Counting infections only from day 20",
    labels={"value": "people", "variable": ""},
).show()
pd.Series(
    {
        "largest difference, two curves above (people)": float(
            np.max(np.abs(frame_c.iloc[:, 0] - frame_c.iloc[:, 1]))
        ),
        "cumulative_integral(start=20) on day 18": float(from_day_20.values[days.get_loc(18.0)]),
        "cumulative_integral(start=20) on day 20": float(from_day_20.values[days.get_loc(20.0)]),
    },
    name="C",
)

> **What to check (C).** In the first figure the two curves are one line and the
> largest difference is at rounding level (well under 0.01 people). In the
> second, `cumulative_integral(start=20)` is 0 up to and including day 20 and
> then follows the fall in S from day 20.

## D. Traced program size against trajectory length

The integral is one `cumsum` over interval areas, with the interval widths
built on the host from the concrete save times. The number of operations in the
jaxpr handed to XLA should be the same for 10 save times as for 10,000.

In [ ]:
def jaxpr_size(n):
    times = np.linspace(0.0, 100.0, n)

    def run(values):
        out = Output(times=TimeAxis(values=times, epoch=None, kind="explicit"), values=values, dims=("time",))
        return out.cumulative_integral(start=float(times[1])).values

    return len(jax.make_jaxpr(run)(jnp.ones(n)).jaxpr.eqns)


sizes = pd.Series({n: jaxpr_size(n) for n in (10, 100, 1_000, 10_000)}, name="jaxpr equations")
sizes.index.name = "save times"
sizes.plot.bar(title="D. jaxpr size of cumulative_integral()", labels={"value": "equations"}).show()
sizes

> **What to check (D).** Every bar is the same height.